# Competition Metadata Investigation

74 of 240 competitions in v5 have **no** `comp_division`, `comp_season_id`, or `comp_season_name`.  
This notebook maps out exactly what we know and don't know about each competition.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 250)

DATA_ROOT = "../../../thesis_data"
v5 = pd.read_parquet(f"{DATA_ROOT}/processed_data/thesis_model_dataset/within_league_transfers_v5.parquet")
master = pd.read_parquet(f"{DATA_ROOT}/processed_data/master_dataset/transfers_model_2018_2025.parquet")

print(f"v5: {v5.shape[0]:,} rows, {v5['from_competition'].nunique()} competitions")
print(f"master: {master.shape[0]:,} rows, {master['from_competition'].nunique()} competitions")

## 1. Competitions WITH metadata

In [ ]:
# Build a competition reference from the master (has comp_name, comp_country, comp_division)
comp_meta = master.groupby("from_competition").agg(
    comp_name=("from_comp_name", "first"),
    comp_country=("from_comp_country", "first"),
    comp_division=("from_comp_division", "first"),
    master_rows=("from_competition", "size"),
).reset_index().rename(columns={"from_competition": "competition_id"})

# v5 competition stats
v5_comp = v5.groupby("from_competition").agg(
    v5_rows=("from_competition", "size"),
    seasons=("from_season", "nunique"),
    season_range=("from_season", lambda x: f"{x.min()}-{x.max()}"),
    teams=("from_team_id", "nunique"),
    division=("from_comp_division", "first"),
    season_name=("from_comp_season_name", "first"),
).reset_index().rename(columns={"from_competition": "competition_id"})

# Merge
comp_info = v5_comp.merge(comp_meta[["competition_id", "comp_name", "comp_country", "master_rows"]],
                          on="competition_id", how="left")

has_meta = comp_info[comp_info["division"].notna()].sort_values("v5_rows", ascending=False)
no_meta = comp_info[comp_info["division"].isna()].sort_values("v5_rows", ascending=False)

print(f"Competitions WITH metadata: {len(has_meta)} ({has_meta['v5_rows'].sum():,} rows, {100*has_meta['v5_rows'].sum()/len(v5):.1f}%)")
print(f"Competitions WITHOUT metadata: {len(no_meta)} ({no_meta['v5_rows'].sum():,} rows, {100*no_meta['v5_rows'].sum()/len(v5):.1f}%)")

In [ ]:
# Full table: competitions WITH metadata
print(f"\n=== {len(has_meta)} COMPETITIONS WITH METADATA ===")
print(f"{'comp_id':>8s} {'v5_rows':>8s} {'seasons':>8s} {'range':>10s} {'teams':>6s} {'div':>4s} {'country':<20s} {'name'}")
print("-" * 100)
for _, r in has_meta.iterrows():
    print(f"{r['competition_id']:>8.0f} {r['v5_rows']:>8d} {r['seasons']:>8d} {r['season_range']:>10s} {r['teams']:>6d} {r['division']:>4.0f} {str(r['comp_country']):<20s} {r['comp_name']}")

In [ ]:
# Summary by division
print("\nBreakdown by division (competitions WITH metadata):")
div_summary = has_meta.groupby("division").agg(
    n_comps=("competition_id", "count"),
    total_rows=("v5_rows", "sum"),
).reset_index()
div_summary["pct_rows"] = (100 * div_summary["total_rows"] / len(v5)).round(1)
print(div_summary.to_string(index=False))

## 2. Competitions WITHOUT metadata — what do we know?

In [ ]:
# For competitions without metadata, check if the master has ANY info
# (comp_name, comp_country were dropped in v2 but exist in master)
print(f"\n=== {len(no_meta)} COMPETITIONS WITHOUT METADATA ===")
print(f"{'comp_id':>8s} {'v5_rows':>8s} {'seasons':>8s} {'range':>10s} {'teams':>6s} {'country':<20s} {'name'}")
print("-" * 90)
for _, r in no_meta.iterrows():
    country = str(r['comp_country']) if pd.notna(r['comp_country']) else '???'
    name = str(r['comp_name']) if pd.notna(r['comp_name']) else '???'
    print(f"{r['competition_id']:>8.0f} {r['v5_rows']:>8d} {r['seasons']:>8d} {r['season_range']:>10s} {r['teams']:>6d} {country:<20s} {name}")

In [ ]:
# Try to get names from the TO side of the master (maybe to_comp_name has data)
to_meta = master.groupby("to_competition").agg(
    to_comp_name=("to_comp_name", "first"),
    to_comp_country=("to_comp_country", "first"),
    to_comp_division=("to_comp_division", "first"),
).reset_index().rename(columns={"to_competition": "competition_id"})

no_meta_enriched = no_meta.merge(to_meta, on="competition_id", how="left")

# Did we recover any info?
recovered = no_meta_enriched[no_meta_enriched["to_comp_name"].notna()]
still_unknown = no_meta_enriched[no_meta_enriched["to_comp_name"].isna()]

print(f"Recovered from TO side: {len(recovered)} comps")
print(f"Still completely unknown: {len(still_unknown)} comps")

if len(recovered) > 0:
    print(f"\nRecovered:")
    print(f"{'comp_id':>8s} {'v5_rows':>8s} {'to_country':<20s} {'to_div':>6s} {'to_name'}")
    print("-" * 80)
    for _, r in recovered.sort_values("v5_rows", ascending=False).iterrows():
        div = f"{r['to_comp_division']:.0f}" if pd.notna(r['to_comp_division']) else '???'
        print(f"{r['competition_id']:>8.0f} {r['v5_rows']:>8d} {str(r['to_comp_country']):<20s} {div:>6s} {r['to_comp_name']}")

In [ ]:
# Last resort: check the raw team stats for these comp IDs
# The team stats parquet might have different coverage
ts = pd.read_parquet(f"{DATA_ROOT}/raw_data/Teams_stats/team_stats_season.parquet")

unknown_comps = no_meta_enriched[no_meta_enriched["to_comp_name"].isna()]["competition_id"].tolist()
if unknown_comps:
    print(f"\nChecking {len(unknown_comps)} unknown comps in raw team_stats:")
    for comp in unknown_comps:
        in_ts = comp in ts["competition_id"].values
        if in_ts:
            n_teams = ts[ts["competition_id"]==comp]["team_id"].nunique()
            n_seasons = ts[ts["competition_id"]==comp]["season"].nunique()
            print(f"  comp={comp}: IN team_stats ({n_teams} teams, {n_seasons} seasons)")
        else:
            print(f"  comp={comp}: NOT in team_stats")

## 3. Summary

In [ ]:
print("=" * 70)
print("COMPETITION METADATA SUMMARY")
print("=" * 70)
print(f"\nTotal competitions in v5: {len(comp_info)}")
print(f"  WITH full metadata (name, country, division): {len(has_meta)} comps, {has_meta['v5_rows'].sum():,} rows ({100*has_meta['v5_rows'].sum()/len(v5):.1f}%)")
print(f"  WITHOUT metadata:                             {len(no_meta)} comps, {no_meta['v5_rows'].sum():,} rows ({100*no_meta['v5_rows'].sum()/len(v5):.1f}%)")

if len(recovered) > 0:
    print(f"\n  Of the {len(no_meta)} without FROM metadata:")
    print(f"    Recovered name from TO side: {len(recovered)}")
    print(f"    Still completely unknown:     {len(still_unknown)}")

print(f"\n--- The 6 comp metadata columns ---")
print(f"  from/to_comp_division:    division tier (1, 2, etc.)")
print(f"  from/to_comp_season_id:   Wyscout internal season ID")
print(f"  from/to_comp_season_name: string like '2022/2023'")
print(f"\n  comp_season_id and comp_season_name are redundant (we have competition + season).")
print(f"  comp_division could be useful but is missing for {len(no_meta)} competitions ({100*no_meta['v5_rows'].sum()/len(v5):.1f}% of rows).")